# Precompute 01 — embed the candidate pool on a Colab GPU

Runs the **same** `src/precompute/01_embed_candidates.py` as locally, but on the T4,
so the full 100k pool takes ~10 min instead of ~14 h on CPU.

**This is offline precompute — GPU + network are allowed here.** `rank.py` never does
any of this; it only loads the artifacts produced. Output = three files
(`candidate_embeddings.npy`, `candidate_ids.npy`, `embeddings_meta.json`).

### The data path (read this once)
Your T4 kernel runs on a **remote Colab VM** — it cannot see your laptop's disk. So the
data must reach the VM and the artifacts must come back. **Google Drive is the channel**
(you're already Google-authenticated via Colab).

**Before running:** upload **one** of these to your Drive root (`MyDrive/`):
* `candidates_compact.jsonl.gz`  ← **recommended, ~9 MB**, identical embedding text
  (staged locally at `redrob-ranker/data/candidates_compact.jsonl.gz`), or
* the full `candidates.jsonl` (487 MB).

Also set **Runtime → Change runtime type → GPU** if not already.

## 1. Confirm the GPU

In [19]:
!nvidia-smi

Fri Jun 26 10:09:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install sentence-transformers
Use Colab's preinstalled **CUDA** torch — do **not** install our `requirements.txt`
(it pins the CPU-only torch for the `rank.py` path).

In [20]:
!pip install -q sentence-transformers

## 3. Get the code (clone the repo onto the VM)

In [21]:
import os

REPO_URL = "https://github.com/CoffeeAurCode/redrob-ranker.git"
if not os.path.isdir("redrob-ranker"):
    !git clone $REPO_URL
%cd redrob-ranker

Cloning into 'redrob-ranker'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 66 (delta 18), reused 59 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 92.18 KiB | 4.85 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/redrob-ranker/redrob-ranker/redrob-ranker/redrob-ranker


## 4. Mount Drive and stage the data
Mounting prompts for a one-time authorization. This auto-detects whichever file you
uploaded (compact `.gz` preferred), unzips if needed, and places it where the script
expects. `wc -l` must print **100000**.

In [22]:
import shutil, subprocess
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"

candidates = [
    f"{DRIVE}/candidates_compact.jsonl.gz",  # recommended (~9 MB)
    f"{DRIVE}/candidates.jsonl.gz",
    f"{DRIVE}/candidates.jsonl",             # full pool
]
src = next((p for p in candidates if os.path.exists(p)), None)
assert src, f"Upload candidates_compact.jsonl.gz (or candidates.jsonl) to {DRIVE}"

os.makedirs("data", exist_ok=True)
dst = "data/candidates.jsonl"
if src.endswith(".gz"):
    subprocess.run(f'gunzip -c "{src}" > {dst}', shell=True, check=True)
else:
    shutil.copy(src, dst)
print("staged from:", src)
!wc -l {dst}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
staged from: /content/drive/MyDrive/candidates_compact.jsonl.gz
100000 data/candidates.jsonl


## 5. Embed the pool (GPU auto-detected)
`SentenceTransformer` picks CUDA automatically. Checkpoints under
`artifacts/.embed_ckpt/`, so if the session drops, re-run this cell to resume.

In [23]:
!python src/precompute/01_embed_candidates.py

2026-06-26 10:09:44,474  INFO  loading model BAAI/bge-base-en-v1.5 …
2026-06-26 10:09:49,708  INFO  NumExpr defaulting to 2 threads.
2026-06-26 10:09:52,529  INFO  TensorFlow version 2.20.0 available.
2026-06-26 10:09:52,530  INFO  JAX version 0.7.2 available.
2026-06-26 10:09:56,718  INFO  No device provided, using cuda:0
2026-06-26 10:09:56,884  WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-26 10:09:56,900  INFO  Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
Loading weights: 100% 199/199 [00:00<00:00, 4571.84it/s]
2026-06-26 10:10:18,483  INFO  chunk 1 done · ~1000 candidates · 50/s
2026-06-26 10:10:39,545  INFO  chunk 2 done · ~2000 candidates · 49/s
2026-06-26 10:11:01,086  INFO  chunk 3 done · ~3000 candidates · 48/s
2026-06-26 10:11:23,608  INFO  chunk 4 done · ~4000 candidates · 47/s
2026-06-26 10:11:47,596  INFO  chunk 5 done · ~5000 candidates · 46/s
202

## 6. Sanity check
Top 30 by similarity to a hand-written "ideal Senior AI Engineer" query. You want
ML / retrieval / ranking / recsys / search / data-science titles near the top — **not**
a keyword-stuffed `Marketing Manager` (that would mean skills are leaking, which they
should not be).

In [24]:
!python src/precompute/sanity_rank.py --top 30

Loading weights: 100% 199/199 [00:00<00:00, 20835.93it/s]

Top 30 by cosine similarity to the "ideal Senior AI Engineer" query:

  #   score  candidate_id    current_title
----------------------------------------------------------------------
  1   0.822  CAND_0077337    Staff Machine Learning Engineer
  2   0.815  CAND_0092278    Senior NLP Engineer
  3   0.815  CAND_0041611    Staff Machine Learning Engineer
  4   0.803  CAND_0081846    Lead AI Engineer
  5   0.799  CAND_0018499    Senior Machine Learning Engineer
  6   0.795  CAND_0002025    Senior AI Engineer
  7   0.791  CAND_0088025    Staff Machine Learning Engineer
  8   0.790  CAND_0046525    Senior Machine Learning Engineer
  9   0.788  CAND_0039754    Senior Applied Scientist
 10   0.785  CAND_0060072    Staff Machine Learning Engineer
 11   0.784  CAND_0094759    Lead AI Engineer
 12   0.775  CAND_0046064    Senior NLP Engineer
 13   0.774  CAND_0011687    Senior NLP Engineer
 14   0.772  CAND_0086022    Senior Applied Scie

## 7. Artifacts in both places: repo `artifacts/` **and** Drive
The embedding script already wrote the three files into the repo's `artifacts/` on the
VM (that is the canonical copy you commit). This cell also copies them to a **separate
Drive folder** `MyDrive/redrob_artifacts/` so you can pull them down. On your machine:
download the Drive copy into your local repo's `artifacts/`, run `pytest -q`, and commit.

In [25]:
OUT = f"{DRIVE}/redrob_artifacts"
os.makedirs(OUT, exist_ok=True)
for name in ("candidate_embeddings.npy", "candidate_ids.npy", "embeddings_meta.json"):
    shutil.copy(f"artifacts/{name}", f"{OUT}/{name}")

print("== repo artifacts/ (canonical copy on the VM clone) ==")
!ls -lh artifacts/*.npy artifacts/*.json
print("\n== Drive copy:", OUT, "==")
!ls -lh "$OUT"
print("\n--- embeddings_meta.json ---")
!cat artifacts/embeddings_meta.json

== repo artifacts/ (canonical copy on the VM clone) ==
-rw-r--r-- 1 root root 147M Jun 26 10:49 artifacts/candidate_embeddings.npy
-rw-r--r-- 1 root root 4.6M Jun 26 10:49 artifacts/candidate_ids.npy
-rw-r--r-- 1 root root  481 Jun 26 10:49 artifacts/embeddings_meta.json

== Drive copy: /content/drive/MyDrive/redrob_artifacts ==
total 152M
-rw------- 1 root root 147M Jun 26 10:49 candidate_embeddings.npy
-rw------- 1 root root 4.6M Jun 26 10:49 candidate_ids.npy
-rw------- 1 root root  481 Jun 26 10:49 embeddings_meta.json

--- embeddings_meta.json ---
{
  "model_id": "BAAI/bge-base-en-v1.5",
  "dim": 768,
  "n": 100000,
  "normalize": true,
  "similarity": "cosine == dot product (vectors L2-normalized)",
  "embeddings_dtype": "float16",
  "passage_prefix": "",
  "query_prefix": "Represent this sentence for searching relevant passages: ",
  "text_source": "profile_text.build_embedding_text (career text; skills excluded)",
  "source": "candidates.jsonl",
  "sentence_transformers_version

In [27]:
import os, json, numpy as np, glob
# We may be at /content or /content/redrob-ranker
for base in ("redrob-ranker/artifacts", "artifacts", "/content/redrob-ranker/artifacts"):
    if os.path.exists(os.path.join(base, "candidate_embeddings.npy")):
        A = base; break
else:
    A = None
print("artifacts dir:", A)
if A:
    emb = np.load(os.path.join(A, "candidate_embeddings.npy"), mmap_mode="r")
    ids = np.load(os.path.join(A, "candidate_ids.npy"))
    meta = json.load(open(os.path.join(A, "embeddings_meta.json")))
    print("embeddings shape/dtype:", emb.shape, emb.dtype)
    print("ids:", ids.shape, ids.dtype, "| unique:", len(set(ids.tolist())) == len(ids))
    print("aligned:", len(ids) == emb.shape[0])
    print("meta:", json.dumps(meta, indent=2))
    for f in ("candidate_embeddings.npy","candidate_ids.npy","embeddings_meta.json"):
        print(f, round(os.path.getsize(os.path.join(A,f))/1e6, 1), "MB")


artifacts dir: artifacts
embeddings shape/dtype: (100000, 768) float16
ids: (100000,) <U12 | unique: True
aligned: True
meta: {
  "model_id": "BAAI/bge-base-en-v1.5",
  "dim": 768,
  "n": 100000,
  "normalize": true,
  "similarity": "cosine == dot product (vectors L2-normalized)",
  "embeddings_dtype": "float16",
  "passage_prefix": "",
  "query_prefix": "Represent this sentence for searching relevant passages: ",
  "text_source": "profile_text.build_embedding_text (career text; skills excluded)",
  "source": "candidates.jsonl",
  "sentence_transformers_version": "5.5.1",
  "created": "2026-06-26"
}
candidate_embeddings.npy 153.6 MB
candidate_ids.npy 4.8 MB
embeddings_meta.json 0.0 MB
